# Project Journal 01: Prototype Snapshot and Real Sleipner Intake Prep

This notebook starts the running project journal for `CO2Shift`. Each future entry should continue the numbered series, for example `02_...`, `03_...`, and so on.


## Context

This stage captured the transition from a bare scaffold to a more credible first-paper prototype.

What already existed before this journal entry:

- a synthetic CCS benchmark with geology families, plume masks, and mismatch perturbations
- classical difference-based baselines
- plain and hybrid ML models for change-map prediction
- uncertainty estimation, saved metrics, and figure outputs

What this stage clarified:

- the hybrid model can now beat the plain ML model on the larger synthetic OOD benchmark
- pseudo-field evaluation is only a plumbing check
- the next serious milestone is real Sleipner export intake through the manifest path


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT = Path.cwd().resolve()
if not (ROOT / 'runs').exists():
    ROOT = ROOT.parent.resolve()

with (ROOT / 'runs' / 'smoke' / 'results' / 'metrics.json').open('r', encoding='utf-8') as handle:
    smoke_metrics = json.load(handle)

with (ROOT / 'runs' / 'paper_proto' / 'results' / 'metrics.json').open('r', encoding='utf-8') as handle:
    proto_metrics = json.load(handle)

ROOT


## What Changed This Stage

The most important changes in this stage were:

- mismatch-aware augmentation was added to hybrid-model training
- reservoir-weighted loss was added to discourage obviously implausible detections
- multi-pair field loading was added through `field.mode: manifest`
- field intake now has a concrete path that can be validated before running a full experiment
- the project now has a journal-series notebook instead of a one-off progress notebook


In [ ]:
def comparison_table(metrics, split_name):
    rows = []
    for method in ['difference', 'impedance', 'plain_ml', 'hybrid_ml']:
        row = {'method': method}
        for key in ['dice', 'iou', 'false_positive_rate', 'ece']:
            row[key] = metrics[split_name][method][key]
        rows.append(row)
    return pd.DataFrame(rows).sort_values('dice', ascending=False).reset_index(drop=True)

print('Smoke test split')
display(comparison_table(smoke_metrics, 'test'))

print('Smoke OOD split')
display(comparison_table(smoke_metrics, 'ood'))

print('Prototype test split')
display(comparison_table(proto_metrics, 'test'))

print('Prototype OOD split')
display(comparison_table(proto_metrics, 'ood'))


## Current Results

The current headline is simple:

- the smoke run proved the pipeline worked
- the larger prototype run gave a more believable result
- on that prototype run, the hybrid model beat the plain ML baseline on both `test` and `ood`

That is strong enough to justify continuing the project, but not strong enough yet to make a field-transfer claim without real exported Sleipner inputs.


In [ ]:
def show_figure(path, title):
    image = mpimg.imread(path)
    plt.figure(figsize=(9, 4))
    plt.imshow(image)
    plt.title(title)
    plt.axis('off')
    plt.show()

show_figure(ROOT / 'runs' / 'paper_proto' / 'results' / 'figures' / 'test_hybrid_example.png', 'Prototype test example')
show_figure(ROOT / 'runs' / 'paper_proto' / 'results' / 'figures' / 'ood_hybrid_example.png', 'Prototype OOD example')
show_figure(ROOT / 'runs' / 'paper_proto' / 'results' / 'figures' / 'field_pseudo_sleipner_hybrid_example.png', 'Prototype pseudo-field example')


## Open Problems

The biggest unresolved issue is not synthetic performance anymore. It is field readiness.

What is still open:

- the project still relies on `pseudo_sleipner` for field-style checks
- outside-reservoir behavior still needs inspection on real exports
- the current model stack expects 2D exported sections, not arbitrary 3D field cubes
- the next round of conclusions should be driven by real manifest-loaded outputs, not more synthetic-only iteration


In [ ]:
field_pairs = proto_metrics.get('field', {}).get('pairs', {})
field_rows = []
for pair_name, pair_metrics in field_pairs.items():
    row = {'pair': pair_name}
    row.update({f'plain_{k}': v for k, v in pair_metrics['plain_ml'].items()})
    row.update({f'hybrid_{k}': v for k, v in pair_metrics['hybrid_ml'].items()})
    field_rows.append(row)

pd.DataFrame(field_rows)


## Next Stage

The next concrete step is real Sleipner intake through the manifest path.

Immediate next actions:

1. Export real baseline and monitor sections as `.npy` or simple single-array `.npz` files.
2. Create a manifest matching the repo template.
3. Run field validation before running a full experiment.
4. Inspect outside-reservoir detections and uncertainty first.

Series note:

- Future journal entries should follow the same structure: `Context`, `What Changed This Stage`, `Current Results`, `Open Problems`, `Next Stage`.
- Future files should continue the numbering convention, such as `02_field_intake.ipynb` and `03_first_field_review.ipynb`.
